In [1]:
import os
from google.colab import drive

drive.mount('/content/drive')
CAMINHO_DATA = '/content/drive/MyDrive/Projeto_Futebol/data'

Mounted at /content/drive


# 2. Data Preparation (Metodologia CRISP-ML)

## 2.1. Objetivos
Nesta etapa, o dataset bruto é limpo, ordenado cronologicamente e enriquecido com **variáveis pré-jogo acumuladas** (médias móveis dos últimos 5 jogos) para garantir que os modelos de aprendizagem automática não utilizem dados do próprio jogo (*data leakage*). O ficheiro resultante é guardado como `dataset_futebol_preparado.csv`.

In [2]:
import os
import glob
import pandas as pd
import numpy as np

def carregar_dados_brutos(caminho_pasta: str) -> pd.DataFrame:
    ficheiros_csv = sorted(glob.glob(os.path.join(caminho_pasta, "*.csv")))
    lista_dfs = []
    for caminho in ficheiros_csv:
        if "dataset_futebol_preparado.csv" in caminho:
            continue
        try:
            df_temp = pd.read_csv(caminho, encoding="utf-8")
        except UnicodeDecodeError:
            df_temp = pd.read_csv(caminho, encoding="latin1")
        df_temp["Epoca"] = os.path.basename(caminho).replace(".csv", "")
        lista_dfs.append(df_temp)
    return pd.concat(lista_dfs, ignore_index=True)

df_bruto = carregar_dados_brutos(CAMINHO_DATA)
print(f"✅ Dados Brutos Carregados: {df_bruto.shape[0]} jogos.")

✅ Dados Brutos Carregados: 8020 jogos.


## 2.2. Seleção de Colunas e Padronização de Datas
Filtra as variáveis estatísticas principais, remove partidas sem resultado e padroniza as datas para o formato `datetime`, ordenando o dataset sequencialmente.

In [3]:
def limpar_e_selecionar_colunas(df: pd.DataFrame) -> pd.DataFrame:
    df_clean = df.copy()

    colunas_essenciais = [
        "Epoca", "Date", "HomeTeam", "AwayTeam",
        "FTHG", "FTAG", "FTR",
        "HTHG", "HTAG", "HTR",
        "HS", "AS", "HST", "AST",
        "HF", "AF", "HC", "AC",
        "HY", "AY", "HR", "AR"
    ]

    colunas_presentes = [col for col in colunas_essenciais if col in df_clean.columns]
    df_clean = df_clean[colunas_presentes]

    df_clean = df_clean.dropna(subset=["HomeTeam", "AwayTeam", "FTR"])
    df_clean["Date"] = pd.to_datetime(df_clean["Date"], dayfirst=True, format="mixed", errors="coerce")
    df_clean = df_clean.dropna(subset=["Date"])

    df_clean = df_clean.sort_values(by=["Date"]).reset_index(drop=True)

    df_clean["HomeTeam"] = df_clean["HomeTeam"].astype(str).str.strip()
    df_clean["AwayTeam"] = df_clean["AwayTeam"].astype(str).str.strip()

    colunas_num = df_clean.select_dtypes(include=[np.number]).columns
    df_clean[colunas_num] = df_clean[colunas_num].fillna(df_clean[colunas_num].median())

    print(f"[LIMPEZA CONCLUÍDA] Registos restantes: {df_clean.shape[0]} | Colunas: {df_clean.shape[1]}")
    return df_clean

## 2.3. Engenharia de Funcionalidades (*Feature Engineering*)
Cria variáveis de desempenho histórico dos **últimos 5 jogos** de cada equipa antes da partida atual:
- Média de Golos Marcados (`L5`)
- Média de Golos Sofridos (`L5`)
- Forma recente em pontos somados (`L5`)

In [4]:
def calcular_features_acumuladas(df: pd.DataFrame, window: int = 5) -> pd.DataFrame:
    df_feat = df.copy()

    def get_points_home(ftr):
        return 3 if ftr == "H" else (1 if ftr == "D" else 0)

    def get_points_away(ftr):
        return 3 if ftr == "A" else (1 if ftr == "D" else 0)

    df_feat["HomePoints"] = df_feat["FTR"].apply(get_points_home)
    df_feat["AwayPoints"] = df_feat["FTR"].apply(get_points_away)

    historico_equipas = {}

    home_gm_l5, away_gm_l5 = [], []
    home_gs_l5, away_gs_l5 = [], []
    home_forma_l5, away_forma_l5 = [], []

    for idx, row in df_feat.iterrows():
        h_team = row["HomeTeam"]
        a_team = row["AwayTeam"]

        for team in [h_team, a_team]:
            if team not in historico_equipas:
                historico_equipas[team] = {"GM": [], "GS": [], "Pts": []}

        h_hist = historico_equipas[h_team]
        a_hist = historico_equipas[a_team]

        home_gm_l5.append(np.mean(h_hist["GM"][-window:]) if len(h_hist["GM"]) > 0 else 0)
        away_gm_l5.append(np.mean(a_hist["GM"][-window:]) if len(a_hist["GM"]) > 0 else 0)

        home_gs_l5.append(np.mean(h_hist["GS"][-window:]) if len(h_hist["GS"]) > 0 else 0)
        away_gs_l5.append(np.mean(a_hist["GS"][-window:]) if len(a_hist["GS"]) > 0 else 0)

        home_forma_l5.append(np.sum(h_hist["Pts"][-window:]) if len(h_hist["Pts"]) > 0 else 0)
        away_forma_l5.append(np.sum(a_hist["Pts"][-window:]) if len(a_hist["Pts"]) > 0 else 0)

        h_hist["GM"].append(row["FTHG"])
        h_hist["GS"].append(row["FTAG"])
        h_hist["Pts"].append(row["HomePoints"])

        a_hist["GM"].append(row["FTAG"])
        a_hist["GS"].append(row["FTHG"])
        a_hist["Pts"].append(row["AwayPoints"])

    df_feat[f"Home_Golos_Marcados_L{window}"] = home_gm_l5
    df_feat[f"Away_Golos_Marcados_L{window}"] = away_gm_l5
    df_feat[f"Home_Golos_Sofridos_L{window}"] = home_gs_l5
    df_feat[f"Away_Golos_Sofridos_L{window}"] = away_gs_l5
    df_feat[f"Home_Forma_L{window}"] = home_forma_l5
    df_feat[f"Away_Forma_L{window}"] = away_forma_l5

    df_feat = df_feat.drop(columns=["HomePoints", "AwayPoints"])
    print(f"[FEATURE ENGINEERING CONCLUÍDO] Janela temporal: {window} jogos.")
    return df_feat

## 2.4. Pipeline de Execução e Exportação Final
Aplica as transformações e guarda o dataset limpo e preparado em `dataset_futebol_preparado.csv`.

In [5]:
# Execução do pipeline
df_limpo = limpar_e_selecionar_colunas(df_bruto)
df_preparado = calcular_features_acumuladas(df_limpo, window=5)

# Ficheiro de saída
caminho_saida = os.path.join(CAMINHO_DATA, "dataset_futebol_preparado.csv")
df_preparado.to_csv(caminho_saida, index=False, encoding="utf-8")

print("\n" + "=" * 70)
print("✅ DATA PREPARATION COMPLETO!")
print(f"Guardado em: {caminho_saida}")
print(f"Dimensões: {df_preparado.shape[0]} linhas x {df_preparado.shape[1]} colunas")
print("=" * 70)

[LIMPEZA CONCLUÍDA] Registos restantes: 8020 | Colunas: 22
[FEATURE ENGINEERING CONCLUÍDO] Janela temporal: 5 jogos.

✅ DATA PREPARATION COMPLETO!
Guardado em: /content/drive/MyDrive/Projeto_Futebol/data/dataset_futebol_preparado.csv
Dimensões: 8020 linhas x 28 colunas
